# 🐶 Bonus Unit 1 — Entrena a Huggy el Perro

**Curso: Aprendizaje por Refuerzo Profundo en Español**  
Adaptación pedagógica basada en el curso de HuggingFace 🤗  
Autor: Jose Miguel Lara Rangel · [aicraft.mx](https://aicraft.mx)

---

### El notebook más visual y motivador del curso

En este notebook vas a entrenar a **Huggy**, un perro virtual en 3D, para que
aprenda a correr y atrapar un palo que le lanzamos. Al terminar podrás
**jugar con él en tiempo real en tu navegador**.

Huggy es especial porque:
- Es el primer entorno del curso con **acciones continuas** — el perro controla
  el ángulo exacto de cada articulación de sus patas
- Nadie le programó cómo caminar — **aprende solo** por prueba y error
- El resultado es visible e interactivo — no solo un número en una tabla

### Cómo se compara Huggy con los entornos anteriores

| Entorno | Estados | Acciones | Tipo de acciones | ¿Interactivo? |
|---------|---------|----------|-----------------|--------------|
| FrozenLake ❄️ | 16 | 4 | Discretas | No |
| CartPole 🏋️ | 4 | 2 | Discretas | No |
| LunarLander 🚀 | 8 | 4 | Discretas | Solo video |
| SnowballTarget 🐻 | ~20 | Continuas | Continuas | Solo video |
| **Huggy 🐶** | **243** | **Articulaciones** | **Continuas** | **¡Sí — en vivo!** |

> 📎 **Slides de referencia:** Capítulo 1.6 del curso.  
> ⏱️ **Tiempo estimado:** ~2.5 horas (setup: ~15 min · entrenamiento: ~2 h · jugar: ilimitado 🎮)

In [ ]:
%%html
<video controls autoplay loop style="width:100%; max-width:640px; border-radius:10px; margin:10px 0;">
  <source src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/notebooks/unit-bonus1/huggy.mp4"
          type="video/mp4">
</video>
<p style="color:gray; font-size:0.85em; text-align:center;">
Huggy aprendiendo a atrapar el palo 🎾 — esto es lo que construiremos hoy</p>

---
## 1 · ¿Cómo aprende Huggy? La teoría antes del código

### 1.1 · La historia de Huggy

Huggy fue creado por **Thomas Simonini** (autor del curso de HF) específicamente para
enseñar RL de forma visual y emocionante. Está inspirado en **Puppo the Corgi**,
un personaje de demostración del equipo de Unity ML-Agents.

A diferencia de los entornos de Atari o Gymnasium — que fueron diseñados para
investigación — Huggy fue diseñado para **enseñar**:
- El objetivo es inmediatamente comprensible: busca el palo
- El éxito es visualmente obvio: el perro corre bien o tropieza
- El resultado es jugable: puedes interactuar con tu agente entrenado

### 1.2 · Las 243 observaciones — qué "siente" Huggy

Huggy no ve con ojos. En cambio, recibe **243 números** en cada paso que describen
su cuerpo y su entorno. Estos números se organizan en 4 grupos:

| Grupo | Nº de valores | Descripción |
|-------|--------------|-------------|
| **Posición del palo** | ~3 | Coordenadas (x, y, z) del palo relativas a Huggy |
| **Orientación de Huggy** | ~3 | En qué dirección está mirando |
| **Estado de articulaciones** | ~200 | Ángulo y velocidad de cada articulación de las 4 patas |
| **Velocidad de la raíz** | ~6 | Velocidad lineal y angular del cuerpo central |

**¿Por qué 243 y no 8 como LunarLander?**

LunarLander solo necesita saber posición, velocidad e inclinación.
Huggy necesita coordinar **docenas de articulaciones simultáneamente** —
cada pata tiene rodilla, cadera y tobillo, y todas deben moverse en sincronía.
Es el mismo desafío que controlar un brazo robótico real.

### 1.3 · Las acciones continuas — la gran diferencia

En todos los entornos anteriores, el agente elegía entre opciones discretas:
izquierda / derecha / arriba / disparar. En Huggy las acciones son **valores continuos**:

```
DQN / Q-Learning:   acción ∈ {0, 1, 2, 3}          ← elige un número entero
PPO para Huggy:     acción ∈ [-1.0, +1.0]^N         ← elige N números reales
```

En cada paso, la política de Huggy produce un vector de ~17 números continuos,
uno por cada motor de articulación:

```python
acción = [-0.23,  0.87, -0.41,  0.62,   # pata delantera izquierda
           0.15, -0.73,  0.28, -0.55,   # pata delantera derecha
          -0.61,  0.44, -0.82,  0.19,   # pata trasera izquierda
           0.38, -0.27,  0.71, -0.48,   # pata trasera derecha
           0.05]                          # rotación del cuerpo
```

Cada número en ese vector le dice a un motor cuánto y en qué dirección girar.
La red neuronal **aprende a coordinar todos estos valores simultáneamente**
para producir un trote natural — sin que nadie le haya explicado cómo caminar.

### 1.4 · La función de recompensa de Huggy

El diseño de la recompensa es lo que guía al perro hacia el comportamiento correcto:

| Recompensa | Tipo | Efecto en el comportamiento |
|-----------|------|----------------------------|
| **+1 × orientación** | Continua | +puntos por mirar hacia el palo — aprende a orientarse |
| **+1 grande** | Terminal | Al alcanzar el palo — la recompensa principal |
| **−1/paso** | Continua | Penalización por tiempo — aprende a correr rápido |
| **0** | — | Al caerse el episodio termina sin recompensa adicional |

Esta combinación es lo que hace que Huggy aprenda a **correr directamente hacia el palo**
en lugar de caminar lentamente en espirales.

### 1.5 · ¿Por qué PPO y no DQN?

DQN necesita calcular Q(s, a) para **todas las acciones posibles** y elegir el máximo.
Con acciones continuas habría infinitas acciones posibles — imposible enumerarlas.

PPO aprende directamente la política π<sub>θ</sub>(a|s), que produce una
**distribución gaussiana** sobre los valores continuos de cada articulación.
No necesita enumerar acciones — muestrea directamente de la distribución.
Esto es exactamente lo que estudiamos en los Capítulos 4.1 y 4.2.

---
## 2 · Configurar el entorno

### 2.1 · GPU obligatoria

Huggy tiene 243 variables de estado y acciones continuas en ~17 dimensiones.
El entrenamiento requiere GPU para terminar en tiempo razonable:

| Configuración | Tiempo para 2M pasos |
|--------------|---------------------|
| Sin GPU (CPU) | ~8–12 horas |
| GPU T4 | ~1.5–2 horas |

**Activar GPU:** Entorno de ejecución → Cambiar tipo → GPU T4

> ⚠️ Si ya completaste el notebook de Unit 5 (ML-Agents), el setup de esta
> sección te resultará familiar. Es exactamente el mismo proceso.
> Puedes ejecutar las celdas directamente sin leerlas en detalle.

In [ ]:
import subprocess
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total',
                         '--format=csv,noheader'], capture_output=True, text=True)
if result.returncode == 0:
    print(f"✅ GPU detectada: {result.stdout.strip()}")
else:
    print("❌ No se detectó GPU — actívala antes de continuar")

### 2.2 · Clonar ML-Agents e instalar dependencias

> Si ya tienes ML-Agents instalado de Unit 5 y sigues en la misma sesión de Colab,
> puedes saltar al paso 2.4.

In [ ]:
%%capture
!git clone --depth 1 https://github.com/Unity-Technologies/ml-agents
print("✅ ML-Agents clonado")

### 2.3 · Ajustar la versión de Python con Miniconda

In [ ]:
!python --version   # mostrar la versión actual

In [ ]:
%%capture
!wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh
!chmod +x Miniconda3-latest-Linux-x86_64.sh
!./Miniconda3-latest-Linux-x86_64.sh -b -f -p /usr/local
!conda install -q -y --prefix /usr/local python=3.10.12 ujson

### 2.4 · Instalar ML-Agents desde el código fuente

In [ ]:
%%capture
%cd ml-agents
!pip3 install -e ./ml-agents-envs
!pip3 install -e ./ml-agents

### 2.5 · Verificar la instalación

In [ ]:
result = subprocess.run(['mlagents-learn', '--help'],
                         capture_output=True, text=True)
if result.returncode == 0 or 'usage' in (result.stdout + result.stderr).lower():
    print("✅ mlagents-learn está listo")
else:
    print("❌ Problema con la instalación — re-ejecuta las celdas 2.3 y 2.4")

---
## 3 · Descargar el entorno de Huggy

El ejecutable de Huggy es un binario de Unity compilado para Linux (~80 MB).
Contiene el motor de física 3D, el modelo del perro, el terreno y la lógica del juego.
No necesitamos Unity instalado — solo ejecutar el binario.

In [ ]:
import os
os.makedirs('./trained-envs-executables/linux', exist_ok=True)
print("✅ Directorio creado")

In [ ]:
# Descargar Huggy (~80 MB)
!wget -q 'https://github.com/huggingface/Huggy/raw/main/Huggy.zip' \
     -O ./trained-envs-executables/linux/Huggy.zip
print("✅ Huggy.zip descargado")

In [ ]:
%%capture
!unzip -d ./trained-envs-executables/linux/ ./trained-envs-executables/linux/Huggy.zip
!chmod -R 755 ./trained-envs-executables/linux/Huggy
print("✅ Ejecutable listo")

In [ ]:
# Verificar
ruta = './trained-envs-executables/linux/Huggy'
if os.path.exists(ruta):
    print(f"✅ Huggy listo en: {ruta}")
    print(f"   Archivos: {os.listdir(ruta)}")
else:
    print("❌ No se encontró el ejecutable. Re-ejecuta las celdas anteriores.")

---
## 4 · El archivo YAML de Huggy — diferencias clave con SnowballTarget

El YAML de Huggy tiene parámetros que no vimos en SnowballTarget.
Aquí los más importantes y por qué cada uno es necesario para un agente de locomoción:

| Parámetro | Huggy | SnowballTarget | ¿Por qué es diferente? |
|-----------|-------|---------------|----------------------|
| `normalize: true` | ✅ | ❌ | Las 243 variables tienen escalas muy distintas (ángulos vs. velocidades) — normalizar las pone en el mismo rango |
| `hidden_units: 512` | 512 | 256 | Red más grande — 243 inputs necesitan más capacidad para aprender patrones de locomoción |
| `num_layers: 3` | 3 | 2 | Una capa extra — la coordinación de articulaciones es más compleja que apuntar y disparar |
| `batch_size: 2048` | 2048 | 128 | Lotes más grandes — acciones continuas tienen mayor varianza, necesitan más datos por actualización |
| `buffer_size: 20480` | 20480 | 2048 | Búfer 10× más grande — más experiencias antes de actualizar |
| `time_horizon: 1000` | 1000 | 64 | Huggy necesita ver consecuencias a largo plazo — un trote de 1000 pasos en lugar de 64 |
| `max_steps: 2e6` | 2M | 200k | Aprender a caminar requiere 10× más pasos que apuntar y disparar |
| `gamma: 0.995` | 0.995 | 0.995 | Igual — ambos necesitan valorar el futuro casi igual que el presente |
| `checkpoint_interval: 200000` | 200k | — | Guarda un checkpoint cada 200k pasos para comparar la evolución |
| `keep_checkpoints: 15` | 15 | 10 | Guarda más checkpoints — la progresión del trote es muy visual |

**La línea más importante: `normalize: true`**

Sin normalización, la red neuronal recibe inputs como:
- Ángulos de articulaciones: entre **−3.14 y +3.14** (radianes)
- Velocidades angulares: entre **−50 y +50** (rad/s)
- Posición del palo: entre **−10 y +10** (metros)

Esas escalas tan distintas hacen que el gradiente sea inestable.
Con `normalize: true`, ML-Agents calcula la media y desviación estándar de cada
variable durante el entrenamiento y las normaliza automáticamente.
Es el equivalente a lo que hacemos manualmente con los retornos en REINFORCE.

In [ ]:
os.makedirs('./config/ppo', exist_ok=True)

huggy_yaml = '''behaviors:
  Huggy:
    trainer_type: ppo
    hyperparameters:
      batch_size: 2048
      buffer_size: 20480
      learning_rate: 0.0003
      beta: 0.005
      epsilon: 0.2
      lambd: 0.95
      num_epoch: 3
      learning_rate_schedule: linear
    network_settings:
      normalize: true
      hidden_units: 512
      num_layers: 3
      vis_encode_type: simple
    reward_signals:
      extrinsic:
        gamma: 0.995
        strength: 1.0
    checkpoint_interval: 200000
    keep_checkpoints: 15
    max_steps: 2000000
    time_horizon: 1000
    summary_freq: 50000
'''

with open('./config/ppo/Huggy.yaml', 'w') as f:
    f.write(huggy_yaml)

print("✅ Huggy.yaml creado en ./config/ppo/")
!cat ./config/ppo/Huggy.yaml

---
## 5 · Entrenar a Huggy

### 5.1 · Lo que vas a observar en los logs

Los logs de Huggy son los más dramáticos del curso. Aquí una guía de qué
esperar en cada etapa del entrenamiento de 2 millones de pasos:

| Etapa | Pasos | Mean Reward | Comportamiento visible |
|-------|-------|-------------|----------------------|
| **Colapso** | 0–200k | ~−5 a 0 | Huggy ni siquiera se sostiene — cae inmediatamente |
| **Gateo** | 200k–600k | 0 a 2 | Empieza a moverse hacia adelante pero tropieza constantemente |
| **Trote torpe** | 600k–1.2M | 2 a 6 | Llega al palo pero con movimientos extraños y asimétricos |
| **Trote natural** | 1.2M–2M | 6 a 10+ | Corre de forma fluida y coordina las 4 patas correctamente |

**¿Por qué empieza en negativo?**
La penalización de −1/paso comienza desde el primer frame.
Al inicio Huggy cae en los primeros milisegundos — episodio muy corto,
pocos pasos, recompensa total ~ −1 a −3.

**La transición más emocionante:** alrededor de los 600k–800k pasos verás
cómo la `Mean Reward` sube de forma casi lineal. Ese es el momento en que
el perro descubre que correr es más eficiente que caminar.

### 5.2 · Lanzar el entrenamiento

> ☕ Este entrenamiento tarda **~1.5–2 horas con GPU T4**.  
> ML-Agents guarda checkpoints cada 200k pasos — si Colab se desconecta,
> puedes continuar desde el último checkpoint con `--resume`.

> 📌 **Monta Google Drive antes de entrenar** si tienes sesión larga:

In [ ]:
# Opcional: montar Google Drive para no perder el progreso
# from google.colab import drive
# drive.mount('/content/drive')
# import shutil
# # Al terminar, copia los resultados:
# shutil.copytree('./results', '/content/drive/MyDrive/rl-course/huggy-results')
print("Descomenta las líneas de arriba si quieres guardar en Google Drive.")

In [ ]:
# Lanzar el entrenamiento de Huggy — 2 millones de pasos
!mlagents-learn ./config/ppo/Huggy.yaml \
    --env=./trained-envs-executables/linux/Huggy/Huggy \
    --run-id='HuggyTraining' \
    --no-graphics

### 5.3 · Si necesitas continuar un entrenamiento interrumpido

Si Colab se desconecta durante el entrenamiento, puedes retomarlo desde el
último checkpoint con el flag `--resume`. El run-id debe ser el mismo:

In [ ]:
# Para continuar desde el último checkpoint (solo si se interrumpió)
# !mlagents-learn ./config/ppo/Huggy.yaml \
#     --env=./trained-envs-executables/linux/Huggy/Huggy \
#     --run-id='HuggyTraining' \
#     --no-graphics \
#     --resume
print("Descomenta y ejecuta si el entrenamiento se interrumpió.")

### 5.4 · Verificar el resultado del entrenamiento

In [ ]:
# Ver los checkpoints guardados
resultado_dir = './results/HuggyTraining'
if os.path.exists(resultado_dir):
    print("✅ Entrenamiento completado. Archivos generados:")
    for root, dirs, files in os.walk(resultado_dir):
        for f in sorted(files):
            ruta = os.path.join(root, f)
            size = os.path.getsize(ruta)
            if size > 1000:
                print(f"  {ruta}  ({size/1024:.1f} KB)")
else:
    print("El directorio de resultados no existe aún.")
    print("Ejecuta el entrenamiento primero.")

---
## 6 · Análisis del aprendizaje motor — ¿cómo emergió el trote?

### ¿Cómo aprendió Huggy a caminar sin que nadie le enseñara?

La locomoción de Huggy es un ejemplo fascinante de **comportamiento emergente**:
nadie programó un algoritmo de marcha, un patrón de pisadas, ni la coordinación
entre patas delanteras y traseras. Todo surgió de maximizar la recompensa.

### La progresión que verás al comparar checkpoints

**~200k pasos — el colapso controlado:**
Huggy aprende primero a no caerse inmediatamente. La red descubre que
mantener el cuerpo horizontal evita la penalización de episodio corto.
El movimiento es extraño — más parecido a arrastrarse que a caminar.

**~600k pasos — el descubrimiento del impulso:**
La red descubre que ciclos de movimiento (repetir el mismo patrón)
son más eficientes energéticamente. Aparece algo parecido a un trote,
aunque asimétrico — a veces usa más las patas izquierdas que las derechas.

**~1.2M pasos — la sincronización:**
Las patas delanteras y traseras empiezan a moverse en fase alternada —
el patrón clásico del trote cuadrúpedo. La red descubrió por cuenta propia
el mismo patrón que usan los perros reales, los caballos y los insectos.

**~2M pasos — la optimización:**
El trote se vuelve fluido y la velocidad aumenta. Huggy aprende a
inclinarse ligeramente hacia adelante al correr — igual que un atleta.

### Conexión con la biología

Los investigadores de robótica han descubierto que los mismos patrones de
marcha que aprende Huggy aparecen en animales reales:
la alternancia de patas diagonales (delantera izquierda + trasera derecha)
es la forma más eficiente de distribuir el peso en cuadrúpedos.
El RL no lo programó — lo **descubrió**.

### Preguntas de reflexión

1. ¿Por qué crees que la red aprendió a sincronizar patas diagonales y no ipsilaterales?
2. Si aumentáramos la penalización por tiempo de −1 a −5 por paso, ¿Huggy correría más rápido o se caería más?
3. ¿Qué pasaría si quitáramos la recompensa de orientación y solo dejáramos la recompensa de llegada?

---
## 7 · 🏋️ Reto: personaliza el comportamiento de Huggy

Ya entrenaste a Huggy con los hiperparámetros estándar. Ahora puedes
experimentar con variaciones que **cambian visiblemente cómo corre el perro**.

> ⚠️ Cada variación re-entrena desde cero. Usa `max_steps: 500000`
> para ver tendencias rápidas sin esperar 2 horas.

### Variación A · Huggy más agresivo (gamma más bajo)

Con `gamma: 0.95` en lugar de `0.995`, Huggy valora mucho menos el futuro.
¿Correría más rápido al principio pero con menos elegancia?
¿Ignoraría la orientación porque el palo está 'demasiado lejos en el futuro'?

In [ ]:
# Variación A — gamma más bajo: Huggy más impaciente
huggy_yaml_a = '''behaviors:
  Huggy:
    trainer_type: ppo
    hyperparameters:
      batch_size: 2048
      buffer_size: 20480
      learning_rate: 0.0003
      beta: 0.005
      epsilon: 0.2
      lambd: 0.95
      num_epoch: 3
      learning_rate_schedule: linear
    network_settings:
      normalize: true
      hidden_units: 512
      num_layers: 3
      vis_encode_type: simple
    reward_signals:
      extrinsic:
        gamma: 0.95
        strength: 1.0
    max_steps: 500000
    time_horizon: 1000
    summary_freq: 50000
'''
with open('./config/ppo/HuggyA.yaml', 'w') as f:
    f.write(huggy_yaml_a)

# Descomenta para entrenar:
# !mlagents-learn ./config/ppo/HuggyA.yaml \
#     --env=./trained-envs-executables/linux/Huggy/Huggy \
#     --run-id='HuggyA_gamma_bajo' \
#     --no-graphics
print("HuggyA.yaml creado. Descomenta el comando para entrenar (~45 min)")

### Variación B · Red más pequeña (¿puede Huggy caminar con menos neuronas?)

Con `hidden_units: 128` y `num_layers: 2` (vs 512 y 3 del original),
la red tiene ~16× menos parámetros. ¿Sigue siendo capaz de coordinar
todas las articulaciones, o el trote se vuelve errático?

In [ ]:
# Variación B — red más pequeña
huggy_yaml_b = '''behaviors:
  Huggy:
    trainer_type: ppo
    hyperparameters:
      batch_size: 2048
      buffer_size: 20480
      learning_rate: 0.0003
      beta: 0.005
      epsilon: 0.2
      lambd: 0.95
      num_epoch: 3
      learning_rate_schedule: linear
    network_settings:
      normalize: true
      hidden_units: 128
      num_layers: 2
      vis_encode_type: simple
    reward_signals:
      extrinsic:
        gamma: 0.995
        strength: 1.0
    max_steps: 500000
    time_horizon: 1000
    summary_freq: 50000
'''
with open('./config/ppo/HuggyB.yaml', 'w') as f:
    f.write(huggy_yaml_b)

# Descomenta para entrenar:
# !mlagents-learn ./config/ppo/HuggyB.yaml \
#     --env=./trained-envs-executables/linux/Huggy/Huggy \
#     --run-id='HuggyB_red_pequeña' \
#     --no-graphics
print("HuggyB.yaml creado. Descomenta el comando para entrenar (~45 min)")

### Variación C · Sin normalización (¿qué pasa si quitamos `normalize: true`?)

Esta es la variación más interesante conceptualmente.
Sin normalización, las 243 variables llegan a la red con escalas radicalmente distintas.
¿Puede la red aprender de todas formas? ¿Cuánto más tarda en converger?

In [ ]:
# Variación C — sin normalización
huggy_yaml_c = '''behaviors:
  Huggy:
    trainer_type: ppo
    hyperparameters:
      batch_size: 2048
      buffer_size: 20480
      learning_rate: 0.0003
      beta: 0.005
      epsilon: 0.2
      lambd: 0.95
      num_epoch: 3
      learning_rate_schedule: linear
    network_settings:
      normalize: false
      hidden_units: 512
      num_layers: 3
      vis_encode_type: simple
    reward_signals:
      extrinsic:
        gamma: 0.995
        strength: 1.0
    max_steps: 500000
    time_horizon: 1000
    summary_freq: 50000
'''
with open('./config/ppo/HuggyC.yaml', 'w') as f:
    f.write(huggy_yaml_c)

# Descomenta para entrenar:
# !mlagents-learn ./config/ppo/HuggyC.yaml \
#     --env=./trained-envs-executables/linux/Huggy/Huggy \
#     --run-id='HuggyC_sin_normalizacion' \
#     --no-graphics
print("HuggyC.yaml creado. Descomenta el comando para entrenar (~45 min)")

### Tabla comparativa de experimentos

Cuando termines de entrenar, documenta tus resultados aquí:

In [ ]:
# Documenta tus resultados
resultados = {
    'Huggy original (2M pasos)':   {'mean_reward': '???', 'comportamiento': 'trote fluido'},
    'Huggy A — gamma bajo (500k)': {'mean_reward': '???', 'comportamiento': '???'},
    'Huggy B — red pequeña (500k)':{'mean_reward': '???', 'comportamiento': '???'},
    'Huggy C — sin normaliz (500k)':{'mean_reward': '???', 'comportamiento': '???'},
}

print("Mis resultados:")
for nombre, datos in resultados.items():
    print(f"  {nombre}")
    print(f"    Mean Reward: {datos['mean_reward']}  |  {datos['comportamiento']}")

---

# 🔒 Sección opcional — Publicar Huggy en el Hub

> Requiere cuenta de HuggingFace.
> Una vez publicado podrás jugar con Huggy en tiempo real desde el navegador.

---

## [OPCIONAL] Publicar tu Huggy en el Hugging Face Hub 🤗

Este es el momento más gratificante del curso: una vez publicado,
puedes ver a tu Huggy correr en el navegador y tirarle el palo.

### Paso 1: Iniciar sesión en HuggingFace

In [ ]:
from huggingface_hub import notebook_login
notebook_login()   # pega tu token con permiso Write

### Paso 2: Publicar el modelo

El comando `mlagents-push-to-hf` sube el modelo `.onnx` al Hub.
Reemplaza `TU_USUARIO` con tu nombre de usuario de HuggingFace
(distingue mayúsculas y minúsculas).

In [ ]:
USERNAME = ''   # ← pon tu usuario aquí

!mlagents-push-to-hf \
    --run-id='HuggyTraining' \
    --local-dir='./results/HuggyTraining' \
    --repo-id='{USERNAME}/ppo-Huggy' \
    --commit-message='Mi Huggy entrenado 🐶🎾'

---
## 8 · ¡A jugar con Huggy! 🎮

### Esta es la sección estrella

Ahora puedes **interactuar con tu agente entrenado en tiempo real**.
Huggy corre en el navegador usando WebGL — no necesitas instalar nada más.

### Cómo acceder al Space interactivo

**Si publicaste tu modelo:**
1. Ve a: **https://huggingface.co/spaces/ThomasSimonini/Huggy**
2. En *Step 1*: escribe tu nombre de usuario (exactamente, distingue mayúsculas)
3. En *Step 2*: selecciona tu repositorio `ppo-Huggy`
4. Haz clic en **Load model**
5. Espera ~30 segundos mientras el modelo carga en el navegador

**Si no publicaste tu modelo** (o quieres ver uno de referencia):
1. Ve a: **https://huggingface.co/spaces/ThomasSimonini/Huggy**
2. Deja el usuario como `ThomasSimonini` (modelo pre-entrenado)
3. Haz clic en **Load model**

### Cómo lanzar el palo

- **Clic en cualquier punto del suelo** → el palo aparece ahí
- Huggy corre hacia el palo automáticamente
- Puedes lanzarlo mientras corre para cambiar su trayectoria
- El episodio se reinicia automáticamente cuando atrapa el palo

### Qué observar mientras juegas

**1. Orientación antes de correr**
Observa que Huggy siempre se orienta hacia el palo antes de acelerar.
Eso es la recompensa de orientación del YAML haciendo su trabajo.

**2. Comportamiento en distancias largas vs. cortas**
Lanza el palo muy cerca y muy lejos. ¿Huggy ajusta su velocidad o siempre corre igual?
¿Por qué crees que ocurre eso?

**3. ¿Qué pasa si mueves el palo mientras corre?**
Haz clic en otro punto mientras Huggy está en movimiento.
¿Puede frenar y reorientarse? ¿Lo hace de forma suave o brusca?

**4. Los límites del aprendizaje**
¿En qué situaciones parece confundirse Huggy?
¿Hay posiciones del palo donde tarda más en llegar o toma una ruta extraña?
¿Qué crees que pasa en el espacio de observaciones cuando el palo está detrás de Huggy?

**5. Comparar con los checkpoints**
Si tienes checkpoints de 500k y 1.5M pasos publicados,
cárgalos y compara el trote visualmente.
¿Cuál es la diferencia más visible entre un Huggy de 500k y uno de 2M pasos?

In [ ]:
%%html
<div style="background: linear-gradient(135deg, #0a0e1a, #0d1520); border: 1px solid rgba(34,211,238,0.3); border-radius: 12px; padding: 20px; margin: 10px 0;">
  <h3 style="color: #22d3ee; margin: 0 0 12px 0;">🎮 Jugar con Huggy en el navegador</h3>
  <p style="color: #94a3b8; margin: 0 0 12px 0;">
    Haz clic en el botón para abrir el Space interactivo de Huggy en una nueva pestaña.
    Necesitas tener tu modelo publicado en el Hub (sección anterior).
  </p>
  <a href="https://huggingface.co/spaces/ThomasSimonini/Huggy" target="_blank"
     style="display: inline-block; background: linear-gradient(90deg, #3b82f6, #22d3ee);
            color: white; padding: 10px 24px; border-radius: 8px; text-decoration: none;
            font-weight: bold; font-size: 1em;">
    🐶 Abrir Huggy en el navegador
  </a>
  <p style="color: #64748b; font-size: 0.85em; margin: 12px 0 0 0;">
    También puedes usar el modelo de referencia: usuario <code>ThomasSimonini</code>
  </p>
</div>

---

## 🎉 ¡Notebook completado!

Huggy aprendió a caminar. Tú aprendiste por qué.

Lo que dominaste en este notebook:

1. **Por qué las acciones continuas son diferentes** — y por qué DQN no sirve aquí
2. **Las 243 observaciones de Huggy** — cómo un perro virtual 'siente' su cuerpo
3. **`normalize: true`** — por qué es crítico con observaciones de escalas distintas
4. **La progresión del aprendizaje motor** — de colapso a trote elegante
5. **Comportamiento emergente** — el patrón de marcha que nadie programó
6. **Experimentación** — cómo pequeños cambios en el YAML cambian el comportamiento

### ¿Qué significa esto más allá del curso?

Las técnicas que usaste con Huggy son las mismas que se usan en:
- **Robots reales** — Boston Dynamics usa variantes de PPO para entrenar a Spot
- **Personajes de videojuegos** — NPC con locomoción aprendida en lugar de animaciones pregrabadas
- **Prótesis biónicas** — control de extremidades artificiales con RL
- **Simulación biomecánica** — entender cómo evolucionó la locomoción en animales

### ¿Qué sigue?

Los Módulos 6, 7 y 8 del curso de HuggingFace cubren:
- **A2C** — Actor-Critic, combinar política y valor
- **Multi-Agent RL** — varios Huggys aprendiendo juntos
- **PPO completo** — la implementación que usa OpenAI para entrenar LLMs

---
*Aicraft · aicraft.mx · Jose Miguel Lara Rangel*